In [7]:
import json
from typing import List, Optional
from pydantic import BaseModel, Field, ValidationError

In [8]:
# 1. Define the target data structure blueprint using Pydantic.
# This class acts as our "fill-in-the-blank" form.
class ExtractedInvoiceData(BaseModel):
    company_name: str = Field(description="The formal name of the business issuing the invoice.")
    invoice_number: int = Field(description="The numerical identifier of the invoice.")
    line_items: List[str] = Field(description="A clean list of all items or services purchased.")
    discount_applied: bool = Field(description="True if a promo code or reduction is present, otherwise False.")
    total_amount_usd: float = Field(description="The final total dollar amount listed on the bill.")
    tax_id: Optional[str] = Field(None, description="The tax identification string, if present.")

In [9]:
# --- Simulation of a Raw System Payload ---
# Imagine a customer uploads this unstructured text snippet into your system:
unstructured_email_text = """
Hey Team, 
I just received the bill from Apex Logistics Solutions for our recent shipment. 
The invoice reference number is 998231. They charged us $1450.00 for the 
Freight Shipping and an extra $75.50 for the Forklift Rental fee. 
They gave us a coupon discount for being a long-term partner, bringing the final total 
owed down to exactly 1525.50 dollars. Let me know when this is paid!
"""

In [10]:
# --- Simulation of an LLM Native Response Layer ---
# When you pass the schema to a modern LLM API, it bypasses normal conversation
# and outputs raw stringified JSON matching your exact variable layout:
mock_llm_json_output = """
{
    "company_name": "Apex Logistics Solutions",
    "invoice_number": 998231,
    "line_items": ["Freight Shipping", "Forklift Rental"],
    "discount_applied": true,
    "total_amount_usd": 1525.50
}
"""

In [11]:
def process_incoming_document(raw_model_output: str):
    """
    Takes the raw string payload outputted by the LLM and validates it 
    against our strict schema rules.
    """
    print("--- Executing Data Validation Layer ---")
    try:
        # Step 1: Parse the raw string text into standard Python dictionaries
        parsed_json = json.loads(raw_model_output)
        
        # Step 2: Pass the dictionary into Pydantic for structural verification.
        # This checks types (e.g., ensuring invoice_number is a strict integer).
        validated_data = ExtractedInvoiceData(**parsed_json)
        
        print("Success! Data matches structural requirements.")
        print(f"Verified Company Name: {validated_data.company_name}")
        print(f"Verified Total Amount: ${validated_data.total_amount_usd}")
        print(f"Line Items Array: {validated_data.line_items}")
        
        # Now this object can safely be sent directly to an SQL database without crashing!
        return validated_data
        
    except ValidationError as e:
        # If the LLM hallucinated a field, forgot a comma, or used a string instead of a float:
        print("\n[CRITICAL ERROR] LLM output broke structural schema contract!")
        print(e.json())
        return None

In [12]:
# Run our validation pipeline
process_incoming_document(mock_llm_json_output)

--- Executing Data Validation Layer ---
Success! Data matches structural requirements.
Verified Company Name: Apex Logistics Solutions
Verified Total Amount: $1525.5
Line Items Array: ['Freight Shipping', 'Forklift Rental']


ExtractedInvoiceData(company_name='Apex Logistics Solutions', invoice_number=998231, line_items=['Freight Shipping', 'Forklift Rental'], discount_applied=True, total_amount_usd=1525.5, tax_id=None)